# Exploring Free Energy Protocol settings

The free energy protocols implemented in **openfe** expose a number of simulation settings that can affect the convergence properties, accuracy and precision of an alchemical transformation. In this notebook, we will explore changes to common settings, advising when you might want to consider changing them and how to change them in practice. For a detailed explanation of the individual protocols and their limitations see the [protocol documentation](https://docs.openfree.energy/en/latest/guide/protocols/) and references therein.


## Sampling and performance settings

This section covers the settings that affect sampling, computational cost and statistical precision.

### Inspecting the default settings

The default settings for each of the protocols are designed to be a starting point for most alchemical transformations with the goal of obtaining reliable estimates with sufficient precision for decision-making at a reasonable computational cost. However, depending on the specific system and transformation being studied, you may want to adjust these settings to improve convergence or precision of the estimation.

The default settings for the `RelativeHybridTopologyProtocol` can be inspected by:

In [1]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()

<div class="alert alert-info">
For a complete explanation of each setting and how to customize the object see <a href="https://docs.openfree.energy/en/latest/cookbook/choose_protocol.html">Choose and Configure a Protocol</a>.
</div>


The main parameters that determine the amount of sampling performed and the associated seeing for the `RelativeHybridTopologyProtocol` are:

| Setting                         | Description                                                                                                              | Setting           |
|---------------------------------|--------------------------------------------------------------------------------------------------------------------------|--------------------------------------------|
| **Lambda windows**              | The number of discrete alchemical states simulated during the transformation.                                            | `lambda_settings.lambda_windows`           |
| **Production length**           | The length of the production simulation performed at each lambda window.                                                 | `simulation_settings.production_length`    |
| **Equilibration length**        | The length of the equilibration simulation performed at each lambda window using the selected multisate sampling method. | `simulation_settings.equilibration_length` |                          |
| **Protocol repeats**            | The number of independent executions of the protocol.                                                                    | `protocol_repeats`                         |

The appropriate changes to these settings are transformation dependent with no single setting guaranteeing sufficient convergence or accuracy in general. These settings provide a good starting point to investigate the affect of additional sampling on the transformation under study however, but will also increase the computational cost of the simulation. The following example retrieves them from the default settings:

In [2]:
n_repeats = settings.protocol_repeats
n_windows = settings.lambda_settings.lambda_windows
production_length = settings.simulation_settings.production_length
equilibration_length = settings.simulation_settings.equilibration_length

print(f"Lambda windows: {n_windows}")
print(f"Production length per state: {production_length.m} ns")
print(f"Equilibration length per state: {equilibration_length.m} ns")
print(f"Independent repeats: {n_repeats}")

Lambda windows: 11
Production length per state: 5.0 ns
Equilibration length per state: 1.0 ns
Independent repeats: 3


### Estimating the computational cost of a protocol

A useful first approximation of the computational cost of a protocol execution is the total simulation time in nanoseconds across all lambda windows and repeats. This can be calculated as:

$$T_{total} = N_{repeats} \times N_{\lambda windows} \times (T_{production} + T_{equilibration})$$

So for the default settings the total simulation time is:

In [5]:
def estimate_rbfe_cost(settings):
    n_repeats = settings.protocol_repeats
    n_windows = settings.lambda_settings.lambda_windows
    production_length = settings.simulation_settings.production_length
    equilibration_length = settings.simulation_settings.equilibration_length

    total_sim_time = n_repeats * n_windows * (production_length + equilibration_length)
    return total_sim_time
total_sim_time_per_leg = estimate_rbfe_cost(settings)
print(f"Total simulation time: {total_sim_time_per_leg.m} ns")

Total simulation time: 198.0 ns


The `RelativeHybridTopologyProtocol` simulates the complex and solvent legs of a transformation independently, so if both use the same settings the total cost of the protocol is double the above estimate. It is useful to consider this estimate when deciding how to adjust the settings for a given transformation.

### Increasing simulation length

This section describes how to increase the amount of production simulation performed at each lambda window and estimate the increase in computational cost.

<div class="alert alert-warning">
<b>Warning:</b> Extending an existing simulation is not currently supported. Increasing the simulation length after running with default settings means the protocol must be run again from the start with the new settings, so the full cost of the longer simulation is incurred.
</div>

#### When to increase simulation length

Consider increasing the simulation length **before** running the simulation when:

- The transformation involves large structural changes due to for example changes in formal charge. In this case you may also need to consider increasing the number of lambda windows see the next section for more information.

Consider increasing the simulation length **after** running the simulation when:

- Free-energy estimates differ substantially between independent repeats of the protocol.
- Forward and reverse free-energy estimates disagree by more than 1kbT at over 50% of the simulation length and their uncertainties do not overlap.[[1]](https://livecomsjournal.org/index.php/livecoms/article/view/v2i1e18378)
- A large number of samples are discarded during analysis due to the automated equilibration and decorrelation methods.
- Replica exchange or MBAR analysis indicates that alchemical overlap or replica mixing is poor between lambda windows, you may also need to consider increasing the number of lambda windows see the next section for more information.

This information is available through the automated analysis performed by the protocols, see the list of available [analysis metrics](https://docs.openfree.energy/en/latest/guide/protocols/relativehybridtopology.html#analysis) for the `RelativeHybridTopologyProtocol` for more information. It is recommended to review the analysis metrics before deciding to increase the simulation length.

#### When not to increase simulation length

Increasing the simulation length is not always the best solution. For example changes to the number of lambda windows or the spacing between them can sometimes be more effective when you have poor overlap between neighboring lambda windows and in some cases a combination of both maybe required.

#### Changing the production length

The following example shows how to double the production simulation length for the `RelativeHybridTopologyProtocol` and then calculates the new total simulation time and compares it to the default settings.


In [8]:
from openff.units import unit

long_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the production length to 10 ns
long_settings.simulation_settings.production_length = 10 * unit.nanoseconds

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=long_settings)

print(f"Updated production length: {protocol.settings.simulation_settings.production_length.m} ns")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg.m} ns")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated production length: 10 ns
New total simulation time: 363.0 ns
Percentage increase in total simulation time: 83.33%


### Increasing the number of lambda windows

Increasing the number of lambda windows introduces additional intermediate alchemical states along the transformation which can make the pathway easier to traverse and improve the overlap between neighboring states. This section describes how to increase the number of lambda windows and estimate the increase in computational cost.

#### When to increase the number of lambda windows

Consider increasing the number of lambda windows **before** running the simulation when:

- The transformation involves a change in net formal charge or substantial redistribution of charge, you may also need to consider increasing the simulation length see the previous section for more information.

Consider increasing the number of lambda windows **after** running the simulation when:

- Neighboring lambda windows have poor phase-space overlap or replica mixing statistics.

#### When not to increase the number of lambda windows

Increasing the number of lambda windows is not always the best solution, slow changes in ligand conformation or binding site rearrangements may require longer simulation lengths to achieve convergence.

#### Changing the number of lambda windows

The following example shows how to double the number of lambda windows for the `RelativeHybridTopologyProtocol` to `22` and calculates the new total simulation time. Note the `lambda_windows` setting only accepts integer values and generates a uniform spacing of lambda values between `0` and `1`.


In [11]:
more_window_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the number of lambda windows to 22
more_window_settings.lambda_settings.lambda_windows = 22
# The number of replicas must be increased to match the number of lambda windows
more_window_settings.simulation_settings.n_replicas = 22

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=more_window_settings)

print(f"Updated number of lambda windows: {protocol.settings.lambda_settings.lambda_windows}")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg.m} ns")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated number of lambda windows: 22
New total simulation time: 396.0 ns
Percentage increase in total simulation time: 100.00%


### Other protocols

The above concepts apply to most free energy protocols, but the settings are located differently. Here we will briefly show how to inspect, and change them for the following protocols:
 - `AbsoluteBindingProtocol`
  - `SepTopProtocol`

These protocols expose similar settings, but they are stored per leg (`complex_*` and `solvent_*`) as those protocols execute both `complex` and `solvent` legs within a single transformation. In addition, each leg has a `*_equil_simulation_settings` object controlling an initial MD simulation used to determine the restraint atoms in the complex legs and seed the solvent legs, which is described in the cost estimate for the `AbsoluteBindingProtocol`.

### AbsoluteBindingProtocol

#### Inspecting the default settings

In [3]:
from openfe.protocols.openmm_afe import AbsoluteBindingProtocol

abfe_default_settings = AbsoluteBindingProtocol.default_settings()

print(f"Independent repeats: {abfe_default_settings.protocol_repeats}")
for leg in ["complex", "solvent"]:
    equil = getattr(abfe_default_settings, f"{leg}_equil_simulation_settings")
    sampling = getattr(abfe_default_settings, f"{leg}_simulation_settings")

    print(f"{leg}:")
    print(f"  Lambda windows: {sampling.n_replicas}")
    print(f"  Production length per state: {sampling.production_length.m} ns")
    print(f"  Equilibration length per state: {sampling.equilibration_length.m} ns")
    print(f"  Pre-equilibration MD length: {(equil.equilibration_length_nvt + equil.equilibration_length + equil.production_length).m} ns")

Independent repeats: 3
complex:
  Lambda windows: 30
  Production length per state: 10.0 ns
  Equilibration length per state: 1 ns
  Pre-equilibration MD length: 5.75 ns
solvent:
  Lambda windows: 14
  Production length per state: 10.0 ns
  Equilibration length per state: 1.0 ns
  Pre-equilibration MD length: 0.8 ns


/Users/joshua/mambaforge/envs/openfe_dev/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


#### Estimating the computational cost

The `AbsoluteBindingProtocol` and `SepTopProtocol` have a different cost profile to the `RelativeHybridTopologyProtocol`. Each leg runs a single, longer MD equilibration (NVT equilibration, NPT equilibration and a production run). This simulation is run once per leg and repeat, **not** once per lambda window, so the total simulation time becomes:

$$T_{total} = N_{repeats} \times \sum_{legs} \left[ T_{MD} + N_{\lambda windows} \times (T_{production} + T_{equilibration}) \right]$$

where $T_{MD}$ is the sum of the NVT equilibration, equilibration and production lengths of the leg's `*_equil_simulation_settings`, and the remaining terms come from its `*_simulation_settings`.

In [1]:
def estimate_abfe_septop_cost(settings):
    abfe_total_sim_time = 0 * unit.nanoseconds
    for leg in ["complex", "solvent"]:
        equil = getattr(settings, f"{leg}_equil_simulation_settings")
        sampling = getattr(settings, f"{leg}_simulation_settings")

        md_time = equil.equilibration_length_nvt + equil.equilibration_length + equil.production_length
        replica_exchange_time = sampling.n_replicas * (sampling.production_length + sampling.equilibration_length)
        leg_time = md_time + replica_exchange_time
        print(f"{leg}: {md_time.m} ns MD + {replica_exchange_time.m} ns replica exchange = {leg_time.m} ns")
        abfe_total_sim_time += leg_time

    abfe_total_sim_time *= settings.protocol_repeats
    print(f"Total simulation time: {abfe_total_sim_time.m:.2f} ns")
    return abfe_total_sim_time

default_abfe_cost = estimate_abfe_septop_cost(abfe_default_settings)

NameError: name 'abfe_default_settings' is not defined

#### Increasing simulation length

The production length is set per leg, so each leg to be extended must be changed individually.

In [9]:
from openfe.protocols.openmm_afe import AbsoluteBindingProtocol

abfe_long_settings = AbsoluteBindingProtocol.default_settings()

# Settings are per leg, so extend both legs
abfe_long_settings.complex_simulation_settings.production_length = 20 * unit.nanoseconds
abfe_long_settings.solvent_simulation_settings.production_length = 20 * unit.nanoseconds

# Create a new protocol object with the updated settings
abfe_protocol = AbsoluteBindingProtocol(settings=abfe_long_settings)

print(f"Updated complex production length: {abfe_protocol.settings.complex_simulation_settings.production_length.m} ns")
print(f"Updated solvent production length: {abfe_protocol.settings.solvent_simulation_settings.production_length.m} ns")

# Calculate the new total simulation time
new_abfe_cost = estimate_abfe_septop_cost(abfe_long_settings)

# Print the percentage increase in total simulation time
percentage_increase = ((new_abfe_cost - default_abfe_cost) / default_abfe_cost) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated complex production length: 20 ns
Updated solvent production length: 20 ns
complex: 5.75 ns MD + 630 ns replica exchange = 635.75 ns
solvent: 0.8 ns MD + 294.0 ns replica exchange = 294.8 ns
Total simulation time: 2791.65 ns
Percentage increase in total simulation time: 89.70%


#### Changing the number of lambda windows

The `AbsoluteBindingProtocol` and `SepTopProtocol` both allow for exact control of the lambda schedule of each leg requiring an explicitly defined list of values, one entry per window. All of the lists for a leg must have the same length, and `n_replicas` in the matching `*_simulation_settings` must be set to the number of windows.

The schedule is defined by `lambda_elec`, `lambda_vdw` and `lambda_restraints`.

In [12]:
abfe_window_settings = AbsoluteBindingProtocol.default_settings()

# Define the new lambda schedule for the complex leg by adding 3 more charge windows
abfe_window_settings.complex_lambda_settings.lambda_elec = [0.0] * 6 + [0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8, 0.85, 0.9] + [1.0] * 15
# Extra vdW windows are added to match the length of the new charge schedule
abfe_window_settings.complex_lambda_settings.lambda_vdw = [0.0] * 19 + [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
# Extra restraint windows are added to match the length of the new charge schedule
abfe_window_settings.complex_lambda_settings.lambda_restraints = [0.0, 0.2, 0.4, 0.6, 0.8] + [1.0] * 28

# The number of replicas must match the number of lambda windows
abfe_window_settings.complex_simulation_settings.n_replicas = len(abfe_window_settings.complex_lambda_settings.lambda_vdw)

# Create a new protocol object with the updated settings
abfe_protocol = AbsoluteBindingProtocol(settings=abfe_window_settings)

# Calculate the new total simulation time
new_abfe_cost = estimate_abfe_septop_cost(abfe_window_settings)

# Print the percentage increase in total simulation time
percentage_increase = ((new_abfe_cost - default_abfe_cost) / default_abfe_cost) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

complex: 5.75 ns MD + 363.0 ns replica exchange = 368.75 ns
solvent: 0.8 ns MD + 154.0 ns replica exchange = 154.8 ns
Total simulation time: 1570.65 ns
Percentage increase in total simulation time: 6.73%


<div class="alert alert-info">
<b>Tip:</b> Changing the number of lambda windows or the lambda schedule for the <b>AbsoluteBindingProtocol</b> and <b>SepTopProtocol</b> by hand can be error prone. The protocols provide a validation method to check that the lambda schedules are valid for the leg in question, use them to validate your changes and catch issues before execution. See the example below.
</div>

In [13]:
# Validate the lambda schedules for the complex leg
AbsoluteBindingProtocol._validate_lambda_schedule(
    abfe_window_settings.complex_lambda_settings,
    abfe_window_settings.complex_simulation_settings
)

### SepTopProtocol

#### Inspecting the default settings

In [4]:
from openfe.protocols.openmm_septop import SepTopProtocol

septop_default_settings = SepTopProtocol.default_settings()

print(f"Independent repeats: {septop_default_settings.protocol_repeats}")
for leg in ["complex", "solvent"]:
    equil = getattr(septop_default_settings, f"{leg}_equil_simulation_settings")
    sampling = getattr(septop_default_settings, f"{leg}_simulation_settings")

    print(f"{leg}:")
    print(f"  Lambda windows: {sampling.n_replicas}")
    print(f"  Production length per state: {sampling.production_length.m} ns")
    print(f"  Equilibration length per state: {sampling.equilibration_length.m} ns")
    print(f"  Pre-equilibration MD length: {(equil.equilibration_length_nvt + equil.equilibration_length + equil.production_length).m} ns")

Independent repeats: 3
complex:
  Lambda windows: 19
  Production length per state: 10.0 ns
  Equilibration length per state: 1.0 ns
  Pre-equilibration MD length: 2.2 ns
solvent:
  Lambda windows: 27
  Production length per state: 10.0 ns
  Equilibration length per state: 1.0 ns
  Pre-equilibration MD length: 2.2 ns


#### Estimating the computational cost

The `SepTopProtocol` has the same cost profile as the `AbsoluteBindingProtocol`, so the `estimate_abfe_septop_cost` function defined above can be reused.

In [7]:
default_septop_cost = estimate_abfe_septop_cost(septop_default_settings)

complex: 2.2 ns MD + 209.0 ns replica exchange = 211.2 ns
solvent: 2.2 ns MD + 297.0 ns replica exchange = 299.2 ns
Total simulation time: 1531.20 ns


#### Increasing simulation length

The production length is changed per leg in the same way as for the `AbsoluteBindingProtocol`.

In [10]:
from openfe.protocols.openmm_septop import SepTopProtocol

septop_long_settings = SepTopProtocol.default_settings()

septop_long_settings.complex_simulation_settings.production_length = 20 * unit.nanoseconds
septop_long_settings.solvent_simulation_settings.production_length = 20 * unit.nanoseconds

# Create a new protocol object with the updated settings
septop_protocol = SepTopProtocol(settings=septop_long_settings)

print(f"Updated complex production length: {septop_protocol.settings.complex_simulation_settings.production_length.m} ns")
print(f"Updated solvent production length: {septop_protocol.settings.solvent_simulation_settings.production_length.m} ns")

# Calculate the new total simulation time
new_septop_cost = estimate_abfe_septop_cost(septop_long_settings)

# Print the percentage increase in total simulation time
percentage_increase = ((new_septop_cost - default_septop_cost) / default_septop_cost) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated complex production length: 20 ns
Updated solvent production length: 20 ns
complex: 2.2 ns MD + 399.0 ns replica exchange = 401.2 ns
solvent: 2.2 ns MD + 567.0 ns replica exchange = 569.2 ns
Total simulation time: 2911.20 ns
Percentage increase in total simulation time: 90.13%


#### Changing the number of lambda windows

The schedule is defined by the `A` and `B` states of each component: `lambda_elec_A`, `lambda_elec_B`, `lambda_vdw_A`, `lambda_vdw_B`, `lambda_restraints_A` and `lambda_restraints_B`. Like the `AbsoluteBindingProtocol` all the lambda value lists for a leg must be the same length.

In [14]:
septop_window_settings = SepTopProtocol.default_settings()

# Define the new lambda schedule for the complex leg adding 3 more charge windows for A and B
septop_window_settings.complex_lambda_settings.lambda_elec_A = [0.0] * 8 + [0.25, 0.4, 0.5, 0.6, 0.75, 0.8] + [1.0] * 8
septop_window_settings.complex_lambda_settings.lambda_elec_B = [1.0] * 8 + [0.8, 0.75, 0.6, 0.5, 0.4, 0.25] + [0.0] * 8
septop_window_settings.complex_lambda_settings.lambda_vdw_A = [0.0] * 15 + [0.14, 0.28, 0.42, 0.57, 0.71, 0.85, 1.0]
septop_window_settings.complex_lambda_settings.lambda_vdw_B = [1.0, 0.85, 0.71, 0.57, 0.42, 0.28, 0.14] + [0.0] * 15
septop_window_settings.complex_lambda_settings.lambda_restraints_A = [0.0, 0.05, 0.1, 0.3, 0.5, 0.75] + [1.0] * 16
septop_window_settings.complex_lambda_settings.lambda_restraints_B = [1.0] * 16 + [0.75, 0.5, 0.3, 0.1, 0.05, 0.0]

# The number of replicas must match the number of lambda windows
septop_window_settings.complex_simulation_settings.n_replicas = len(septop_window_settings.complex_lambda_settings.lambda_vdw_A)

# Create a new protocol object with the updated settings
septop_protocol = SepTopProtocol(settings=septop_window_settings)

# Calculate the new total simulation time
new_septop_cost = estimate_abfe_septop_cost(septop_window_settings)

# Print the percentage increase in total simulation time
percentage_increase = ((new_septop_cost - default_septop_cost) / default_septop_cost) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

complex: 2.2 ns MD + 242.0 ns replica exchange = 244.2 ns
solvent: 2.2 ns MD + 297.0 ns replica exchange = 299.2 ns
Total simulation time: 1630.20 ns
Percentage increase in total simulation time: 6.47%


In [15]:
# Validate the lambda schedules for the complex leg
SepTopProtocol._validate_lambda_schedule(
    septop_window_settings.complex_lambda_settings,
    septop_window_settings.complex_simulation_settings
)